<a href="https://colab.research.google.com/github/24Sinchana/project/blob/main/ML%20PROJECTS/Movie_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import difflib  #find close match of the movie name or values (compare)
from sklearn.feature_extraction.text import TfidfVectorizer #convert text to numerical data
from sklearn.metrics.pairwise import cosine_similarity # gives similarity score



In [4]:
#data collection and preprocessing
#loading data from csv file to pandas adateframe
movies_data=pd.read_csv("movies.csv")

In [5]:
movies_data.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [6]:
movies_data.shape

(4803, 24)

In [7]:
movies_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

In [8]:
movies_data.isnull().sum()

,0
index,0
budget,0
genres,28
homepage,3091
id,0
keywords,412
original_language,0
original_title,0
overview,3
popularity,0


In [9]:
#selecting relavent features
selected_features=['genres','keywords','tagline','cast','director']
print(selected_features)

['genres', 'keywords', 'tagline', 'cast', 'director']


In [10]:
#replace null values with null string
for feature in selected_features:
    movies_data[feature]=movies_data[feature].fillna('')

In [11]:
movies_data.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [12]:
movies_data.isnull().sum()

,0
index,0
budget,0
genres,0
homepage,3091
id,0
keywords,0
original_language,0
original_title,0
overview,3
popularity,0


In [14]:
#combing all selected_features
combined_features=movies_data['genres']+' '+movies_data['keywords']+' '+movies_data['tagline']+' '+movies_data['cast']+ ''+movies_data['director']

In [15]:
print(combined_features)

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states\u2013mexic...
4799    Comedy Romance  A newlywed couple's honeymoon ...
4800    Comedy Drama Romance TV Movie date love at fir...
4801      A New Yorker in Shanghai Daniel Henney Eliza...
4802    Documentary obsession camcorder crush dream gi...
Length: 4803, dtype: object


In [16]:
#converting text dat to feature vectors
vectorizer=TfidfVectorizer()


In [17]:
feature_vector=vectorizer.fit_transform(combined_features)

In [18]:
print(feature_vector)

  (0, 208)	0.07667385108184376
  (0, 289)	0.08800104830707552
  (0, 6241)	0.1083630860656862
  (0, 16345)	0.10110130418423781
  (0, 6437)	0.10110130418423781
  (0, 4357)	0.2086788913449303
  (0, 3615)	0.21664083919579374
  (0, 6923)	0.1606391519765174
  (0, 17262)	0.33130376319063676
  (0, 19787)	0.12241864673585882
  (0, 3806)	0.24348426965339803
  (0, 17147)	0.2086788913449303
  (0, 5865)	0.23437015044361997
  (0, 18275)	0.06921925370543773
  (0, 20389)	0.12507033381973537
  (0, 13517)	0.08827534630310274
  (0, 13860)	0.26544401841252097
  (0, 16062)	0.15564527461690644
  (0, 20398)	0.23860058795491665
  (0, 20721)	0.19702892145320783
  (0, 16029)	0.21898867021233073
  (0, 16879)	0.20091311266965015
  (0, 19900)	0.19827148152003798
  (0, 17532)	0.16181995525842374
  (0, 10559)	0.22730069036139045
  :	:
  (4801, 423)	0.19056341009568284
  (4801, 5734)	0.2563183496274323
  (4801, 20694)	0.29933113097807007
  (4801, 16621)	0.28905348514055085
  (4801, 20559)	0.3138166592040448
  (4801, 

In [19]:
# Cosine similarity-getting similarity scores
similarity=cosine_similarity(feature_vector)

In [20]:
print(similarity)

[[1.         0.07047992 0.01423497 ... 0.         0.         0.        ]
 [0.07047992 1.         0.03271772 ... 0.03557523 0.         0.        ]
 [0.01423497 0.03271772 1.         ... 0.         0.02997687 0.        ]
 ...
 [0.         0.03557523 0.         ... 1.         0.         0.02941525]
 [0.         0.         0.02997687 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02941525 0.         1.        ]]


In [21]:
print(similarity.shape)

(4803, 4803)


In [22]:
#index and similarity score

In [23]:
#getting movie name from user
movie_name=input("enter your favourite movie name:")

enter your favourite movie name:iron man


In [24]:
#creating list of all movies name
list_of_all_titles=movies_data['title'].tolist() #creates a list

In [25]:
print(list_of_all_titles)

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz: The Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lant

In [27]:
#finding close match for the movie name given by user
find_close_match=difflib.get_close_matches(movie_name,list_of_all_titles)

In [28]:
print(find_close_match)

['Iron Man', 'Iron Man 3', 'Iron Man 2']


In [31]:
close_match=find_close_match[0]
print(close_match)  #most relavent

Iron Man


In [34]:
#find the corresponding index of the movie with title
index_of_movie = movies_data[movies_data.title == close_match]['index'].values[0]
print(index_of_movie)

68


In [49]:
#getting list of similar movies
#enumerate to carry loop in the list
similarity_score=list(enumerate(similarity[index_of_movie]))
print(similarity_score)

[(0, 0.03229579644860707), (1, 0.05393195039447942), (2, 0.013495256913644414), (3, 0.006290137712561313), (4, 0.03181599223363476), (5, 0.013492855232607511), (6, 0.0807002229078615), (7, 0.23948092153801243), (8, 0.007641303920257968), (9, 0.07528320175535286), (10, 0.07440697853473807), (11, 0.011816684330342166), (12, 0.01349314196279947), (13, 0.01219543022207434), (14, 0.09532110167793535), (15, 0.007191559327707054), (16, 0.22696359695614313), (17, 0.012738836763260195), (18, 0.040336420677614754), (19, 0.07763620585984603), (20, 0.07702266637203237), (21, 0.011076120703301915), (22, 0.006787153718668808), (23, 0.006402612273657428), (24, 0.01226292406425624), (25, 0.0), (26, 0.21748863690185471), (27, 0.029945938769572685), (28, 0.06190026789378868), (29, 0.013589845716434311), (30, 0.07814636293349293), (31, 0.27155401360493714), (32, 0.028052126365086392), (33, 0.129125151923067), (34, 0.0), (35, 0.03417250378858465), (36, 0.031252233451102195), (37, 0.007885689399278514), (3

In [50]:
#first number represents the index of the movie ,second value represents the similarity score of iron man movie with other movie

In [51]:
len(similarity_score)

4803

In [52]:
#sorting based on similarity score
sorted_similar_movies=sorted(similarity_score, key = lambda x:x[1],reverse=True)

In [53]:
print(sorted_similar_movies) #decending order

[(68, 1.0000000000000004), (79, 0.348539886661646), (31, 0.27155401360493714), (7, 0.23948092153801243), (16, 0.22696359695614313), (26, 0.21748863690185471), (85, 0.20384498126302353), (182, 0.19211495786414462), (511, 0.16526444635810034), (64, 0.15008682959582387), (203, 0.14651794298050327), (174, 0.14326369370437603), (4401, 0.1430567313047247), (46, 0.1399174245717579), (101, 0.13929773066794668), (169, 0.13514452526336213), (788, 0.13122776344970674), (94, 0.13105605324769884), (126, 0.1309077781595354), (33, 0.129125151923067), (3623, 0.11773832986774722), (2442, 0.11576168742539061), (353, 0.11301836236268734), (131, 0.10937368345102097), (38, 0.10906527471967553), (1740, 0.10886177866608818), (122, 0.10626607781771327), (1451, 0.10557107626215012), (242, 0.1041908336844519), (618, 0.10261732891660269), (1210, 0.09932679833929622), (954, 0.09917190541875923), (2390, 0.098307771574909), (3166, 0.09732062356161367), (3385, 0.09710030708125605), (2235, 0.09697177344765094), (1406

In [57]:
#key is x=similarity score x[0]=first element that is index
#print the name of similar movies based on the index

print("Movies suggested for you:")
i=1
#getting movie name by index
for movie in sorted_similar_movies:

    index=movie[0] #represent first value(0,1.0000)
    title_from_index=movies_data[movies_data.index==index]['title'].values[0]
    if(i<30):   #first 30 similar movies
        print(i,'.',title_from_index)
        i+=1


Movies suggested for you:
1 . Iron Man
2 . Iron Man 2
3 . Iron Man 3
4 . Avengers: Age of Ultron
5 . The Avengers
6 . Captain America: Civil War
7 . Captain America: The Winter Soldier
8 . Ant-Man
9 . X-Men
10 . X-Men: Apocalypse
11 . X2
12 . The Incredible Hulk
13 . The Helix... Loaded
14 . X-Men: Days of Future Past
15 . X-Men: First Class
16 . Captain America: The First Avenger
17 . Deadpool
18 . Guardians of the Galaxy
19 . Thor: The Dark World
20 . X-Men: The Last Stand
21 . Made
22 . Southland Tales
23 . Tropic Thunder
24 . G-Force
25 . The Amazing Spider-Man 2
26 . Kick-Ass 2
27 . X-Men Origins: Wolverine
28 . Zoom
29 . Fantastic Four


Movie Recommendation System

In [58]:
movie_name=input("enter your favourite movie name:")

list_of_all_titles=movies_data['title'].tolist()

find_close_match=difflib.get_close_matches(movie_name,list_of_all_titles)
close_match=find_close_match[0]
index_of_movie = movies_data[movies_data.title == close_match]['index'].values[0]
similarity_score=list(enumerate(similarity[index_of_movie]))

sorted_similar_movies=sorted(similarity_score, key = lambda x:x[1],reverse=True)


print("Movies suggested for you:")
i=1
for movie in sorted_similar_movies:

    index=movie[0] #represent first value(0,1.0000)
    title_from_index=movies_data[movies_data.index==index]['title'].values[0]
    if(i<30):   #first 30 similar movies
        print(i,'.',title_from_index)
        i+=1


enter your favourite movie name:bat man
Movies suggested for you:
1 . Batman
2 . Batman Returns
3 . Batman & Robin
4 . The Dark Knight Rises
5 . Batman Begins
6 . The Dark Knight
7 . A History of Violence
8 . Superman
9 . Bedazzled
10 . Man of Steel
11 . Suicide Squad
12 . Beetlejuice
13 . The Postman Always Rings Twice
14 . Salton Sea
15 . Jonah Hex
16 . Spider-Man 3
17 . Superman Returns
18 . Mars Attacks!
19 . Spider-Man 2
20 . Exorcist II: The Heretic
21 . Reds
22 . Superman III
23 . Superman II
24 . Something's Gotta Give
25 . Multiplicity
26 . I Dreamed of Africa
27 . Catwoman
28 . Batman Forever
29 . The One
